In [ ]:
# Problema: El equipo de operaciones necesita decidir si un extracto máquina-día cumple el contrato antes de usarlo para sus indicadores.

import json
from pathlib import Path

import pandas as pd

ACTIVITY_DIR = Path('..').resolve()
DATA_DIR = ACTIVITY_DIR / 'data'
SUBMISSION_DIR = ACTIVITY_DIR / 'submission'


In [ ]:
EXPECTED_COLUMNS = (
    'factory_id',
    'machine_id',
    'daily_units_produced',
    'factory_date',
)
BUSINESS_KEY = ('factory_id', 'machine_id', 'factory_date')


def validate_data(dataframe):
    violations = []

    if tuple(dataframe.columns) != EXPECTED_COLUMNS:
        return ['El esquema no coincide con el contrato esperado.']
    if not dataframe['factory_id'].gt(0).all():
        violations.append('factory_id debe contener valores positivos.')
    if not dataframe['machine_id'].gt(0).all():
        violations.append('machine_id debe contener valores positivos.')
    if not dataframe['daily_units_produced'].ge(0).all():
        violations.append('daily_units_produced no puede ser negativo.')
    if pd.to_datetime(
        dataframe['factory_date'], format='%Y-%m-%d', errors='coerce'
    ).isna().any():
        violations.append('factory_date debe contener fechas válidas.')
    if dataframe.duplicated(BUSINESS_KEY).any():
        violations.append('La llave factory_id-machine_id-factory_date está duplicada.')

    return violations


In [ ]:
dataset_names = [
    'machine_throughput_export.csv',
    'invalid_negative_production.csv',
    'invalid_duplicate_key.csv',
    'invalid_date.csv',
    'invalid_schema.csv',
]

results = []
for dataset_name in dataset_names:
    dataframe = pd.read_csv(DATA_DIR / dataset_name)
    violations = validate_data(dataframe)
    results.append(
        {
            'dataset': dataset_name,
            'rows': len(dataframe),
            'accepted': not violations,
            'violations': violations,
        }
    )

pd.DataFrame(results)[['dataset', 'rows', 'accepted', 'violations']]


In [ ]:
report = {
    'contract_columns': list(EXPECTED_COLUMNS),
    'business_key': list(BUSINESS_KEY),
    'datasets': results,
}

SUBMISSION_DIR.mkdir(exist_ok=True)
report_path = SUBMISSION_DIR / 'validation_report.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')

print(f'Reporte generado: {report_path.name}')
